# Come and Don't Come: Same Math, Different Starting State

Come bets look like a new category of wager on the craps table, but
mathematically they are pass line bets in disguise — identical
house edge, identical per-roll probabilities, identical expected
value once you realize the dice have no memory.

This notebook walks through the no-memory argument, cross-checks
it against the exact `Fraction` values from `craps_lab.probability`,
and visualizes the equivalence with a seeded Monte Carlo so you
can see pass-line and come-bet running edges overlaying each other
byte for byte.

## What is a come bet?

Come bets are placed **after** a point has been established on the
table, during the point phase of a regular shooter's round. When
you place a come bet, the *next* roll becomes its own personal
come-out:

- 7 or 11 → come bet wins immediately
- 2, 3, or 12 → come bet loses immediately
- 4, 5, 6, 8, 9, or 10 → the bet "travels" to that number, which
  becomes its **come point**. The come bet then wins if the shooter
  rolls the come point before rolling 7, and loses if 7 comes first.

Don't come is the mirror image with the bar-12 rule, exactly as
don't pass mirrors pass line.

Structurally: a come bet is a pass line bet whose come-out is
shifted to whatever the next roll happens to be, and whose point
phase overlays whatever table state already exists. The **rules**
of the bet are unchanged.

## Why the house edge is identical

The dice have no memory. Given any table state — point established
or not, which point, how many rolls ago the point was set — the
distribution of the next roll is still just the 2d6 PMF,

$$
P(S = s) \;=\; \frac{\mathrm{count}(s)}{36},
$$

independent of history. So the game tree **rooted at the moment of
placement** is identical for:

- a pass line bet placed before the come-out roll, and
- a come bet placed during the point phase.

Both face a come-out decision (naturals win, craps lose, else
establish a point), then both face a point-phase decision
($p$ wins, 7 loses) with the same $P(p \text{ before } 7)$
probabilities. Identical game trees have identical expected
values, and therefore identical house edges:

$$
P(\text{come win}) \;=\; P(\text{pass win}) \;=\; \frac{244}{495}, \qquad
\text{edge}(\text{come}) \;=\; \text{edge}(\text{pass}) \;=\; \frac{7}{495}.
$$

The same argument gives $P(\text{don't come win}) = P(\text{don't pass win}) = 949/1980$ and $\text{edge}(\text{don't come}) = \text{edge}(\text{don't pass}) = 3/220$ under the bar-12 rule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from craps_lab.bets import Outcome
from craps_lab.dice import DiceRoller
from craps_lab.play import (
    play_come_bet,
    play_dont_come_bet,
    play_dont_pass,
    play_pass_line,
)
from craps_lab.probability import (
    come_bet_house_edge,
    come_bet_win_probability,
    dont_come_bet_house_edge,
    dont_come_bet_win_probability,
    dont_pass_house_edge,
    dont_pass_win_probability,
    pass_line_house_edge,
    pass_line_win_probability,
)

plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.grid": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 0xC0C0

In [ ]:
print("=== pass line vs. come bet ===")
print(f"P(pass win)         = {pass_line_win_probability()}")
print(f"P(come bet win)     = {come_bet_win_probability()}")
print(f"Equal as Fraction?  {pass_line_win_probability() == come_bet_win_probability()}")
print()
print(f"pass line edge      = {pass_line_house_edge()}  = {float(pass_line_house_edge()):.6f}")
print(f"come bet edge       = {come_bet_house_edge()}  = {float(come_bet_house_edge()):.6f}")
print(f"Equal as Fraction?  {pass_line_house_edge() == come_bet_house_edge()}")

print()
print("=== don't pass vs. don't come ===")
print(f"P(don't pass win)       = {dont_pass_win_probability()}")
print(f"P(don't come win)       = {dont_come_bet_win_probability()}")
print(f"Equal as Fraction?      {dont_pass_win_probability() == dont_come_bet_win_probability()}")
print()
print(f"don't pass edge  = {dont_pass_house_edge()}  = {float(dont_pass_house_edge()):.6f}")
print(f"don't come edge  = {dont_come_bet_house_edge()}  = {float(dont_come_bet_house_edge()):.6f}")
print(f"Equal as Fraction?  {dont_pass_house_edge() == dont_come_bet_house_edge()}")

## Seeded Monte Carlo comparison

As a belt-and-suspenders check, roll through 50,000 games of each
bet under the same seed and plot the running empirical edges. The
come-bet curves should overlay their line-bet counterparts *exactly*
— not approximately — because the play runners in `craps_lab.play`
delegate at the function level, producing byte-identical outcome
sequences.

In [ ]:
MAX_GAMES = 50_000


def running_edge(play_func, seed):
    roller = DiceRoller(seed=seed)
    trace = np.zeros(MAX_GAMES)
    wins = 0
    losses = 0
    for i in range(MAX_GAMES):
        outcome = play_func(roller)
        if outcome == Outcome.WIN:
            wins += 1
        elif outcome == Outcome.LOSE:
            losses += 1
        trace[i] = (losses - wins) / (i + 1)
    return trace


pass_trace = running_edge(play_pass_line, SEED)
come_trace = running_edge(play_come_bet, SEED)
dont_pass_trace = running_edge(play_dont_pass, SEED + 1)
dont_come_trace = running_edge(play_dont_come_bet, SEED + 1)

n_axis = np.arange(1, MAX_GAMES + 1)

fig, (ax_pass, ax_dont) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

ax_pass.plot(n_axis, pass_trace, color="steelblue", linewidth=1.2, label="pass line empirical")
ax_pass.plot(n_axis, come_trace, color="tomato", linewidth=1.2, linestyle="--", label="come bet empirical")
ax_pass.axhline(
    float(pass_line_house_edge()),
    color="black",
    linewidth=0.8,
    linestyle=":",
    label=f"analytical = 7/495",
)
ax_pass.set_xscale("log")
ax_pass.set_xlabel("games played")
ax_pass.set_ylabel("empirical house edge")
ax_pass.set_title("pass line vs. come bet")
ax_pass.legend(loc="upper right", fontsize=9)

ax_dont.plot(n_axis, dont_pass_trace, color="steelblue", linewidth=1.2, label="don't pass empirical")
ax_dont.plot(n_axis, dont_come_trace, color="tomato", linewidth=1.2, linestyle="--", label="don't come empirical")
ax_dont.axhline(
    float(dont_pass_house_edge()),
    color="black",
    linewidth=0.8,
    linestyle=":",
    label=f"analytical = 3/220",
)
ax_dont.set_xscale("log")
ax_dont.set_xlabel("games played")
ax_dont.set_title("don't pass vs. don't come")
ax_dont.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

print(f"\npass empirical edge     @50k: {pass_trace[-1]:.6f}")
print(f"come bet empirical edge @50k: {come_trace[-1]:.6f}")
print(f"Byte-identical: {pass_trace[-1] == come_trace[-1]}")
print()
print(f"don't pass empirical edge     @50k: {dont_pass_trace[-1]:.6f}")
print(f"don't come bet empirical edge @50k: {dont_come_trace[-1]:.6f}")
print(f"Byte-identical: {dont_pass_trace[-1] == dont_come_trace[-1]}")

## So why do come bets exist at all?

If the math is identical, why would anyone place a come bet instead
of another pass line bet? Two reasons:

1. **Multiple simultaneous positions.** Only one pass line bet can
   be active per round (on the come-out). A come bet can be placed
   on any roll during the point phase, so a player can have several
   come bets "working" on different numbers at once. That isn't an
   edge advantage — the house still charges 1.414% — but it's an
   *exposure* lever: more money on the table per shooter, which
   matters for bankroll management and variance budgeting.
2. **Fast-track into the point phase.** A come bet skips the
   come-out waiting period. If the shooter has already set a point
   and is deep into the phase, a new pass line bet cannot be placed
   until the next round; a come bet can be placed right now.

Neither reason changes the house edge. The only strategic value is
in how come bets interact with **odds bets**, the topic of the next
notebook.

## Takeaways

1. **Come bet house edge = pass line house edge = $7/495 \approx 1.414\%$**, *exactly* — same `Fraction`, not approximately equal.
2. **Don't come edge = don't pass edge = $3/220 \approx 1.364\%$**, exactly, under the bar-12 rule.
3. The equivalence is the **no-memory property** of the dice: the
   game tree rooted at the moment a come bet is placed is identical
   to the game tree rooted at a pass line come-out, so expected
   values and variances match exactly.
4. The Monte Carlo confirms this by construction: the running
   edges overlay byte-for-byte because the play runners delegate
   at the function level.

**Next up:** **free odds bets** — the rare wager with *zero* house
edge, which attaches to an existing line or come bet and dilutes
the composite edge to below 0.6% at 3-4-5x maximum odds. That is
where come bets start earning their strategic keep.